# 8. Psycholinguistic analysis with LIWC (Section 4.5)

Brazilian Portuguese LIWC 2007 dictionary. For each dataset the raw text is split with NLTK's
`word_tokenize` (lowercase; NLTK stopwords and punctuation removed) and every token is looked up in
the dictionary (wildcard entries such as `abafad*` match by prefix). Each category is compared
between FW.Br and FT.Br with a 2x2 chi-square test; phi is the effect size and p-values are
adjusted with Holm-Bonferroni.

In [ ]:
from utils import *
import nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)

In [ ]:
dictionary = load_liwc_dictionary()
count_wa, tokens_wa = liwc_count(load_dataset("whatsapp")["text_content_anonymous"], dictionary)
count_tg, tokens_tg = liwc_count(load_dataset("telegram")["text_content_anonymous"], dictionary)
print(f"tokens: FW.Br {tokens_wa:,} | FT.Br {tokens_tg:,}")

In [ ]:
rows = []
for category in sorted(set(count_wa) | set(count_tg)):
    a, b = count_wa[category], count_tg[category]
    chi2, p, _, _ = stats.chi2_contingency([[a, tokens_wa - a], [b, tokens_tg - b]], correction=False)
    p_wa, p_tg = a / tokens_wa, b / tokens_tg
    se = np.sqrt(p_wa * (1 - p_wa) / tokens_wa + p_tg * (1 - p_tg) / tokens_tg)
    diff = p_tg - p_wa
    rows.append({"category": category, "% FW.Br": 100 * p_wa, "% FT.Br": 100 * p_tg,
                 "difference (pp)": 100 * diff, "CI low (pp)": 100 * (diff - 1.96 * se),
                 "CI high (pp)": 100 * (diff + 1.96 * se), "p": p,
                 "phi": np.sqrt(chi2 / (tokens_wa + tokens_tg))})

liwc = pd.DataFrame(rows)
liwc["p (Holm)"] = holm(liwc["p"])
liwc = liwc.sort_values("difference (pp)", key=abs, ascending=False)
liwc.to_csv(results_folder("08_liwc") / "liwc_comparison.csv", index=False)

print(f"categories: {len(liwc)} | significant: {(liwc['p'] <= 0.05).sum()} | "
      f"after Holm: {(liwc['p (Holm)'] <= 0.05).sum()}")
print(f"phi: max {liwc['phi'].max():.3f}, median {liwc['phi'].median():.3f}")
liwc.head(12).round(4)

## Comparison with the paper

In [ ]:
row = liwc.set_index("category").reindex(["time", "relig", "humans", "money", "we", "anger", "anx"])
compare_with_paper([
    ("categories compared", 63, len(liwc)),
    ("significant (p <= 0.05)", 49, (liwc["p"] <= 0.05).sum()),
    ("significant after Holm", 40, (liwc["p (Holm)"] <= 0.05).sum()),
    ("phi max", 0.033, round(liwc["phi"].max(), 3)),
    ("phi median", 0.008, round(liwc["phi"].median(), 3)),
    ("time % FW.Br", 6.87, round(row.loc["time", "% FW.Br"], 2)), ("time % FT.Br", 8.77, round(row.loc["time", "% FT.Br"], 2)),
    ("relig % FW.Br", 12.20, round(row.loc["relig", "% FW.Br"], 2)), ("relig % FT.Br", 10.11, round(row.loc["relig", "% FT.Br"], 2)),
    ("humans % FW.Br", 13.92, round(row.loc["humans", "% FW.Br"], 2)), ("humans % FT.Br", 11.85, round(row.loc["humans", "% FT.Br"], 2)),
    ("money % FW.Br", 2.50, round(row.loc["money", "% FW.Br"], 2)), ("money % FT.Br", 3.38, round(row.loc["money", "% FT.Br"], 2)),
    ("we % FW.Br", 0.74, round(row.loc["we", "% FW.Br"], 2)), ("we % FT.Br", 0.41, round(row.loc["we", "% FT.Br"], 2)),
    ("anger p", 0.41, round(row.loc["anger", "p"], 2)), ("anx p", 0.46, round(row.loc["anx", "p"], 2)),
])